# 08 — Model Dataset (v_2 — All Competitions)

Join player qualities (pre/post) with team qualities (from projected + to current) into one modelling parquet.

**Inputs:**
- `player_qualities_pre_post.parquet` — ~95k internal transfers, 20 pre + 20 post player qualities
- `team_stats_z_scores_qualities.parquet` — ~22k team-season rows, 7 current + 7 projected team qualities

**Expected:** Lower join match rate than v_1 (~91% vs 99%) since some competitions in transfers may not pass the MIN_TEAMS threshold in team stats.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_ROOT = '/Users/jorgepadilla/Documents/Documents - Jorge\u2019s MacBook Air/thesis_data'
V2 = f'{DATA_ROOT}/processed_data/model_dataset/v_2'

players = pd.read_parquet(f'{V2}/player_qualities_pre_post.parquet')
teams = pd.read_parquet(f'{V2}/team_stats_z_scores_qualities.parquet')

print(f'Players: {players.shape}')
print(f'Teams:   {teams.shape}')
print(f'Player competitions: {players["competition_id"].nunique()}')
print(f'Team competitions:   {teams["competition_id"].nunique()}')

Players: (86438, 51)
Teams:   (21800, 240)
Player competitions: 274
Team competitions:   310


## 1. Prepare Team Quality Subsets

In [2]:
# From-team: projected qualities (q_proj_*)
q_proj_cols = [c for c in teams.columns if c.startswith('q_proj_')]
from_team = teams[['team_id', 'competition_id', 'season'] + q_proj_cols].copy()
from_team = from_team.rename(columns={c: f'from_{c}' for c in q_proj_cols})
from_team = from_team.rename(columns={'team_id': 'from_team_id', 'season': 'from_season'})

# To-team: current qualities (q_*)
q_curr_cols = [c for c in teams.columns if c.startswith('q_') and not c.startswith('q_proj_')]
to_team = teams[['team_id', 'competition_id', 'season'] + q_curr_cols].copy()
to_team = to_team.rename(columns={c: f'to_{c}' for c in q_curr_cols})
to_team = to_team.rename(columns={'team_id': 'to_team_id', 'season': 'to_season'})

print(f'From-team: {from_team.shape}')
print(f'To-team:   {to_team.shape}')

From-team: (21800, 10)
To-team:   (21800, 10)


## 2. Join

In [3]:
n0 = len(players)

# Join from-team projected qualities
df = players.merge(
    from_team,
    on=['from_team_id', 'competition_id', 'from_season'],
    how='left',
)
n1 = len(df)

# Join to-team current qualities
df = df.merge(
    to_team,
    on=['to_team_id', 'competition_id', 'to_season'],
    how='left',
)
n2 = len(df)

print(f'Before joins:      {n0:,}')
print(f'After from-team:   {n1:,}')
print(f'After to-team:     {n2:,}')

if n2 != n0:
    print(f'WARNING: Row count changed! Possible duplicate team-season keys.')

Before joins:      86,438
After from-team:   86,438
After to-team:     86,438


## 3. Join Coverage Diagnostics

In [4]:
from_proj_cols = [c for c in df.columns if c.startswith('from_q_proj_')]
to_curr_cols = [c for c in df.columns if c.startswith('to_q_')]

from_nulls = df[from_proj_cols].isna().any(axis=1).sum()
to_nulls = df[to_curr_cols].isna().any(axis=1).sum()
either_null = df[from_proj_cols + to_curr_cols].isna().any(axis=1).sum()

print(f'Rows missing from-team qualities: {from_nulls:,} ({from_nulls/len(df):.1%})')
print(f'Rows missing to-team qualities:   {to_nulls:,} ({to_nulls/len(df):.1%})')
print(f'Rows missing either:              {either_null:,} ({either_null/len(df):.1%})')

# Diagnostic: which competitions lose the most rows?
null_mask = df[from_proj_cols + to_curr_cols].isna().any(axis=1)
if null_mask.sum() > 0:
    print(f'\nRows lost by competition (top 15):')
    lost = df[null_mask].groupby('comp_name').size().sort_values(ascending=False).head(15)
    for name, n in lost.items():
        total = (df['comp_name'] == name).sum()
        print(f'  {name}: {n:,} / {total:,} ({n/total:.0%})')

# Drop rows with missing team qualities
df_clean = df.dropna(subset=from_proj_cols + to_curr_cols)
print(f'\nAfter dropping nulls: {len(df_clean):,} ({len(df_clean)/len(df):.1%} retained)')

Rows missing from-team qualities: 7,542 (8.7%)
Rows missing to-team qualities:   4,967 (5.7%)
Rows missing either:              8,207 (9.5%)

Rows lost by competition (top 15):
  J2 League: 386 / 956 (40%)
  J1 League: 361 / 920 (39%)
  CSL: 291 / 695 (42%)
  Primera División: 277 / 2,886 (10%)
  China League One: 224 / 493 (45%)
  UAE Pro League: 191 / 485 (39%)
  K League 1: 183 / 519 (35%)
  Premier League: 151 / 4,074 (4%)
  Serie A: 142 / 1,250 (11%)
  Qatar Stars League: 129 / 338 (38%)
  Primera Division: 126 / 609 (21%)
  V.League 1: 123 / 371 (33%)
  Indian Super League: 120 / 344 (35%)
  Liga Profesional de Fútbol: 106 / 662 (16%)
  Liga Pro: 102 / 582 (18%)

After dropping nulls: 78,231 (90.5% retained)


## 4. Select & Order Final Columns

In [5]:
PLAYER_QUALITIES = [
    'Active defence', 'Aerial threat', 'Box threat', 'Chance prevention',
    'Composure', 'Defensive heading', 'Dribbling', 'Effectiveness',
    'Finishing', 'Hold-up play', 'Intelligent defence', 'Involvement',
    'Passing quality', 'Poaching', 'Pressing', 'Progression',
    'Providing teammates', 'Run quality', 'Territorial dominance', 'Winning duels',
]

TEAM_QUALITIES = ['DEFENCE', 'DEFENSIVE_TRANSITION', 'ATTACKING_TRANSITION',
                  'ATTACK', 'PENETRATION', 'CHANCE_CREATION', 'OUTCOME']

id_cols = [
    'wy_player_id',
    'from_team_id', 'to_team_id',
    'competition_id', 'comp_name',
    'from_season', 'to_season',
    'position',
    'player_season_age',
    'pre_minutes', 'post_minutes',
]

from_team_cols = [f'from_q_proj_{q}' for q in TEAM_QUALITIES]
to_team_cols = [f'to_q_{q}' for q in TEAM_QUALITIES]
pre_player_cols = [f'pre_{q}' for q in PLAYER_QUALITIES]
post_player_cols = [f'post_{q}' for q in PLAYER_QUALITIES]

# Compute deltas
delta_team_cols = []
for q in TEAM_QUALITIES:
    col = f'delta_team_{q}'
    df_clean[col] = df_clean[f'to_q_{q}'] - df_clean[f'from_q_proj_{q}']
    delta_team_cols.append(col)

delta_player_cols = []
for q in PLAYER_QUALITIES:
    col = f'delta_{q}'
    df_clean[col] = df_clean[f'post_{q}'] - df_clean[f'pre_{q}']
    delta_player_cols.append(col)

all_cols = (id_cols + from_team_cols + to_team_cols + delta_team_cols
            + pre_player_cols + post_player_cols + delta_player_cols)

missing = [c for c in all_cols if c not in df_clean.columns]
if missing:
    print(f'WARNING: Missing columns: {missing}')
else:
    print(f'All {len(all_cols)} columns present')

model_df = df_clean[all_cols].reset_index(drop=True)

print(f'\nFinal shape: {model_df.shape}')
print(f'\n  IDs & context:              {len(id_cols)}')
print(f'  From-team projected quals:  {len(from_team_cols)}')
print(f'  To-team current quals:      {len(to_team_cols)}')
print(f'  Delta team quals:           {len(delta_team_cols)}')
print(f'  Pre-transfer player quals:  {len(pre_player_cols)}')
print(f'  Post-transfer player quals: {len(post_player_cols)}')
print(f'  Delta player quals:         {len(delta_player_cols)}')

All 92 columns present

Final shape: (78231, 92)

  IDs & context:              11
  From-team projected quals:  7
  To-team current quals:      7
  Delta team quals:           7
  Pre-transfer player quals:  20
  Post-transfer player quals: 20
  Delta player quals:         20


## 5. Summary Statistics

In [6]:
print(f'Transfers: {len(model_df):,}')
print(f'Unique players: {model_df["wy_player_id"].nunique():,}')
print(f'Competitions: {model_df["competition_id"].nunique()}')
print(f'Positions: {sorted(model_df["position"].unique())}')
print(f'Windows: {sorted(model_df["to_season"].unique())}')

print(f'\nTransfers by position:')
print(model_df['position'].value_counts().to_string())

print(f'\nTransfers by window:')
print(model_df['to_season'].value_counts().sort_index().to_string())

print(f'\nTop 10 competitions:')
top = model_df.groupby('comp_name').size().sort_values(ascending=False).head(10)
for name, n in top.items():
    print(f'  {name}: {n:,}')

Transfers: 78,231
Unique players: 42,531
Competitions: 228
Positions: ['Central Defender', 'Full Back', 'Midfielder', 'Striker', 'Winger']
Windows: [2020, 2021, 2022, 2023, 2024]

Transfers by position:
position
Midfielder          23448
Central Defender    19226
Full Back           16476
Winger              10187
Striker              8894

Transfers by window:
to_season
2020    10645
2021    12755
2022    17554
2023    17497
2024    19780

Top 10 competitions:
  Premier League: 3,923
  Primera División: 2,609
  First League: 1,810
  Championship: 1,288
  Pro League: 1,280
  Bundesliga: 1,200
  MLS: 1,161
  Superliga: 1,111
  Serie A: 1,108
  Ligue 1: 1,070


In [7]:
import plotly.express as px

# Quality coverage by position
print('PLAYER QUALITY NULL RATES BY POSITION')
for pos in sorted(model_df['position'].unique()):
    mask = model_df['position'] == pos
    pre_null = model_df.loc[mask, pre_player_cols].isna().mean().mean()
    post_null = model_df.loc[mask, post_player_cols].isna().mean().mean()
    print(f'  {pos:20s} pre={pre_null:.1%}  post={post_null:.1%}  n={mask.sum():,}')

print(f'\nTEAM QUALITY STATS')
print(model_df[from_team_cols + to_team_cols].describe().round(3).to_string())

PLAYER QUALITY NULL RATES BY POSITION
  Central Defender     pre=4.6%  post=4.6%  n=19,226
  Full Back            pre=2.4%  post=2.2%  n=16,476
  Midfielder           pre=13.6%  post=13.5%  n=23,448
  Striker              pre=10.5%  post=10.7%  n=8,894
  Winger               pre=10.1%  post=10.1%  n=10,187

TEAM QUALITY STATS
       from_q_proj_DEFENCE  from_q_proj_DEFENSIVE_TRANSITION  from_q_proj_ATTACKING_TRANSITION  from_q_proj_ATTACK  from_q_proj_PENETRATION  from_q_proj_CHANCE_CREATION  from_q_proj_OUTCOME  to_q_DEFENCE  to_q_DEFENSIVE_TRANSITION  to_q_ATTACKING_TRANSITION  to_q_ATTACK  to_q_PENETRATION  to_q_CHANCE_CREATION  to_q_OUTCOME
count            78231.000                         78231.000                         78231.000           78231.000                78231.000                    78231.000            78231.000     78231.000                  78231.000                  78231.000    78231.000         78231.000             78231.000     78231.000
mean                 0

## 6. Save

In [8]:
# Save ALL (same-team + different-team)
out_all = f'{V2}/model_df.parquet'
model_df.to_parquet(out_all, index=False)

# Save TRANSFERS ONLY (different team)
transfers_only = model_df[model_df['from_team_id'] != model_df['to_team_id']].reset_index(drop=True)
out_transfers = f'{V2}/model_df_transfers_only.parquet'
transfers_only.to_parquet(out_transfers, index=False)

print(f'=== ALL (same-team + transfers) ===')
print(f'Saved: {out_all}')
print(f'Shape: {model_df.shape}')
print(f'Competitions: {model_df["competition_id"].nunique()}')

print(f'\n=== TRANSFERS ONLY (different team) ===')
print(f'Saved: {out_transfers}')
print(f'Shape: {transfers_only.shape}')
print(f'Competitions: {transfers_only["competition_id"].nunique()}')

print(f'\nBy position:')
for pos in sorted(model_df['position'].unique()):
    n_all = (model_df['position'] == pos).sum()
    n_tr = (transfers_only['position'] == pos).sum()
    print(f'  {pos:20s}  all={n_all:>6,}  transfers={n_tr:>6,}  ({n_tr/n_all:.0%})')

=== ALL (same-team + transfers) ===
Saved: /Users/jorgepadilla/Documents/Documents - Jorge’s MacBook Air/thesis_data/processed_data/model_dataset/v_2/model_df.parquet
Shape: (78231, 92)
Competitions: 228

=== TRANSFERS ONLY (different team) ===
Saved: /Users/jorgepadilla/Documents/Documents - Jorge’s MacBook Air/thesis_data/processed_data/model_dataset/v_2/model_df_transfers_only.parquet
Shape: (20467, 92)
Competitions: 216

By position:
  Central Defender      all=19,226  transfers= 4,733  (25%)
  Full Back             all=16,476  transfers= 4,240  (26%)
  Midfielder            all=23,448  transfers= 5,842  (25%)
  Striker               all= 8,894  transfers= 3,031  (34%)
  Winger                all=10,187  transfers= 2,621  (26%)
